In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [26]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [27]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192112 entries, 0 to 192111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        192112 non-null  int64 
 1   profile    192112 non-null  object
 2   anime_uid  192112 non-null  int64 
 3   score      192112 non-null  int64 
 4   scores     192112 non-null  object
 5   text       192112 non-null  object
dtypes: int64(3), object(3)
memory usage: 8.8+ MB


In [29]:
df.shape

(192112, 6)

In [30]:
df['target'] = df['score'].apply(lambda x : 'Good' if x >= 5 else "Bad")

In [31]:
df['target']

0         Good
1         Good
2         Good
3         Good
4         Good
          ... 
192107    Good
192108    Good
192109     Bad
192110    Good
192111    Good
Name: target, Length: 192112, dtype: object

In [32]:
df['target'].value_counts()

target
Good    169994
Bad      22118
Name: count, dtype: int64

In [33]:
df

,uid,profile,anime_uid,score,scores,text,target
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...,Good
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...,Good
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...,Good
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...,Good
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...,Good
...,...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...,Good
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...,Good
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...,Bad
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...,Good


In [34]:
df_majority = df[df['target'] == 'Good']
df_minority = df[df['target'] == 'Bad']

In [35]:
df_reduced = df_majority.sample(n=len(df_minority), random_state=42)

In [36]:
df_balanced = pd.concat([df_reduced, df_minority]).sample(frac=1, random_state=42).reset_index()

In [37]:
df_balanced

,index,uid,profile,anime_uid,score,scores,text,target
0,90145,65641,rachel-chanx3,11739,9,"{'Overall': '9', 'Story': '10', 'Animation': '...",we still can t imagine our futures or foresee ...,Good
1,147133,232715,Dead_or_H3ntai,12055,4,"{'Overall': '4', 'Story': '4', 'Animation': '5...",i ll be quick about this read the brandish man...,Bad
2,80111,177049,Valkqt,17729,3,"{'Overall': '3', 'Story': '2', 'Animation': '6...",be it for the dorky humour or the sporadic bou...,Bad
3,129819,254924,xgreeneyednekox,34501,4,"{'Overall': '4', 'Story': '3', 'Animation': '5...",in short entertaining but don t expect too muc...,Bad
4,176820,304469,Blood_Diver_A,37302,8,"{'Overall': '8', 'Story': '7', 'Animation': '6...",this review contains spoilers a seriously unde...,Good
...,...,...,...,...,...,...,...,...
44231,115802,202516,justa333,4722,9,"{'Overall': '9', 'Story': '7', 'Animation': '5...",the reason i enjoyed this anime is because of ...,Good
44232,173615,299148,Artrill,37450,5,"{'Overall': '5', 'Story': '5', 'Animation': '3...",5 0 10 a blas boy stands on a bridge overlooki...,Good
44233,135172,160993,literaturenerd,1280,2,"{'Overall': '2', 'Story': '2', 'Animation': '2...",overview we have enough people on this site re...,Bad
44234,114811,207548,Xembled,30948,7,"{'Overall': '7', 'Story': '7', 'Animation': '7...",i know a lot of people don t really enjoy this...,Good


In [38]:
x, y = df_balanced['text'], df_balanced['target']

In [39]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(le.classes_)

['Bad' 'Good']


In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [41]:
word_tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=10,
    ngram_range=(1,2),
    sublinear_tf=True
)

In [42]:
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    max_features=4000,
    ngram_range=(3,5),
    sublinear_tf=True,
    min_df=10
)

In [43]:
tfidf = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

In [44]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [45]:
model = MultinomialNB(
    alpha=1.0
)

model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [46]:
y_pred = model.predict(X_test_tfidf)

In [47]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      4424
           1       0.86      0.83      0.84      4424

    accuracy                           0.85      8848
   macro avg       0.85      0.85      0.85      8848
weighted avg       0.85      0.85      0.85      8848



In [48]:
print(confusion_matrix(y_test, y_pred))

[[3831  593]
 [ 772 3652]]
